In [101]:
import numpy as np
import os
import sys
import gc
import glob
import warnings
import geopandas as gpd
from pyhdf.SD import SD, SDC
from multiprocessing import Pool
from pyproj import Transformer
from shapely.geometry import mapping
from matplotlib.path import Path

warnings.filterwarnings("ignore")

data_dir = "/Volumes/project/mgreenst/energy_flux"

# Keep only pixels in produced grids
grid = gpd.read_file(f"{data_dir}/intermediate/jx_grid.gpkg", layer='grid').dissolve()
geom = grid.geometry.iloc[0]
coords = np.array(mapping(geom)['coordinates'][0])
poly_path = Path(coords)

# Fast bbox filter
minx, miny, maxx, maxy = geom.bounds



In [111]:
coords

array([[ 186494.17271798, 2840794.88631329],
       [ 186494.17271798, 2845794.88631329],
       [ 186494.17271798, 2850794.88631329],
       ...,
       [ 191494.17271798, 2835794.88631329],
       [ 191494.17271798, 2840794.88631329],
       [ 186494.17271798, 2840794.88631329]], shape=(611, 2))

In [ ]:
year = 2024
total_valid_cot = 0
total_either_fill = 0
files_checked = 0

day=101
cloud_paths = sorted(glob.glob(
    f"{data_dir}/modis_l2/MOD06_L2/{year}/{str(day).zfill(3)}/MOD06_L2.A{year}{str(day).zfill(3)}.*.hdf"
))
cp = cloud_paths[30]
    




/Volumes/project/mgreenst/energy_flux/modis_l2/MOD06_L2/2024/101/MOD06_L2.A2024101.0230.061.2024101133745.hdf


In [129]:
# Each worker needs its own transformer
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32650", always_xy=True)

# Construct corresponding path for MOD03 geolocation file
cp_geo_list = glob.glob(cp.replace("MOD06_L2", "MOD03")[:-17] + "*.hdf")

cp_geo = cp_geo_list[0]
hdf = SD(cp, SDC.READ)
hdf_geo = SD(cp_geo, SDC.READ)

# Read geolocation
lat_sds = hdf_geo.select('Latitude')
lon_sds = hdf_geo.select('Longitude')
lat = lat_sds[:].astype('float32').flatten()
lon = lon_sds[:].astype('float32').flatten()
lat_fill = lat_sds.attributes()['_FillValue']
lon_fill = lon_sds.attributes()['_FillValue']

# Mark fill values as NaN
lat[lat == lat_fill] = np.nan
lon[lon == lon_fill] = np.nan

# Convert to EPSG:32650
xs, ys = transformer.transform(lon, lat)
bbox_mask = np.isfinite(xs) & np.isfinite(ys) & (xs >= minx) & (xs <= maxx) & (ys >= miny) & (ys <= maxy)

bbox_mask

array([False, False, False, ..., False, False, False], shape=(2748620,))

In [110]:
# Precise containment
points = np.column_stack([xs[bbox_mask], ys[bbox_mask]])
points

array([[ 644406.08723906, 3330740.42138408],
       [ 645755.95307349, 3330481.88033403],
       [ 638342.26313015, 3330696.78172941],
       ...,
       [ 642295.58206599, 2951072.14610607],
       [ 644071.90965723, 2950722.58775807],
       [ 645775.27910927, 2950387.64945875]], shape=(137111, 2))

In [118]:
inside = poly_path.contains_points(points)
mask_1km = bbox_mask.copy()
mask_1km[bbox_mask] = inside
mask_1km

array([False, False, False, ..., False, False, False], shape=(2748620,))

In [121]:
# Read COT
cot_sds = hdf.select('Cloud_Optical_Thickness')
cot = cot_sds[:].astype('float32').flatten()[mask_1km]
cot_fill = cot_sds.attributes()['_FillValue']

cot_fill
cot

array([-9999., -9999., -9999., ..., -9999.,   131.,   143.],
      shape=(94466,), dtype=float32)

In [122]:
# Read Cloud Top Temperature 1km
ctt_sds = hdf.select('cloud_top_temperature_1km')
ctt = ctt_sds[:].astype('float32').flatten()[mask_1km]
ctt_fill = ctt_sds.attributes()['_FillValue']
ctt_fill

-999

In [132]:
# Read Cloud Top Pressure 1km
ctp_sds = hdf.select('Cloud_Optical_Thickness_Uncertainty')
ctt = ctp_sds[:].astype('float32').flatten()[mask_1km]
ctt

array([-9999., -9999., -9999., ..., -9999.,   776.,   555.],
      shape=(94466,), dtype=float32)

In [124]:
hdf.end()
hdf_geo.end()

In [125]:
# Count
valid_cot = cot != cot_fill
valid_cot

array([False, False, False, ..., False,  True,  True], shape=(94466,))

In [126]:
n_valid = int(valid_cot.sum())

In [ ]:
n_either = int((valid_cot & ((ctt == ctt_fill) | (ctp == ctp_fill))).sum())

In [128]:
n_either

0